# Day 3, hands-on 2: which dataset would you trust, worked

Two profiles of the same shape of file, and one question: which one would you compute on?

Every placeholder is filled with the option the answer key records, and the notebook is executed
from a clean kernel so every output and every check is visible on the page. The line under each
step says why the other three letters fail.

Where this sits in the day, and the steps this notebook walks.

In [1]:
import sys, pathlib

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts"))
        break
import c2kit as kit

kit.side_by_side(
    kit.ladder(["profiling the columns", "the rows a profile cannot see", "hands-on: the full pass", "hands-on: which dataset"], lit=3, title="the day's notebooks", show=False),
    kit.flow(["read profile A", "read profile B", "compare them", "say which and why"], title="this notebook's steps", show=False),
)

## Setup

Two profiles, built from the same seven fields. One is the file you profiled today. The other is what it looks like after somebody helpfully coerced everything.

In [2]:
orders = kit.load_csv("C2_W01_D03_orders_STUDENT.csv")

def profile(rows, field):
    return {"present": sum(1 for r in rows if r[field].strip()),
            "converts": sum(1 for r in rows if r[field].strip().lstrip("-").isdigit()),
            "distinct": len({r[field] for r in rows})}

A = {f: profile(orders, f) for f in orders[0]}
coerced = [{**r, "amount": r["amount"] if r["amount"].strip().lstrip("-").isdigit() else "0",
            "discount": r["discount"] or "0"} for r in orders]
B = {f: profile(coerced, f) for f in coerced[0]}

kit.table(["field", "A present", "A converts", "A distinct", "B present", "B converts", "B distinct"],
          [[f, A[f]["present"], A[f]["converts"], A[f]["distinct"],
            B[f]["present"], B[f]["converts"], B[f]["distinct"]] for f in A])

field,A present,A converts,A distinct,B present,B converts,B distinct
order_id,50,0,49,50,0,49
customer_id,50,0,47,50,0,47
segment,50,0,4,50,0,4
amount,48,44,46,50,50,42
status,50,0,3,50,0,3
order_date,50,0,20,50,0,20
discount,11,11,9,50,50,9


## Step 1. Read profile A

A is the raw file. Say what its amount column is telling you before you look at B at all.

In [3]:
kit.flow(["read profile A", "read profile B", "compare them", "say which and why"], lit=0)

In [4]:
# TODO 1. How many amount values are present and unusable in A?
#   a) A["amount"]["present"] - A["amount"]["converts"]
#   b) 50 - A["amount"]["converts"]
#   c) A["amount"]["distinct"] - A["amount"]["converts"]
#   d) A["amount"]["present"]
gap_a = A["amount"]["present"] - A["amount"]["converts"]
print("A amount:", A["amount"], "present and unusable:", gap_a)

A amount: {'present': 48, 'converts': 44, 'distinct': 46} present and unusable: 4


In [5]:
kit.check("four amounts are present and unusable in A", gap_a == 4, f"{gap_a} values")
kit.check("the two empty cells are absent rather than unusable",
          A["amount"]["present"] == 48)

Fifty minus converts counts the two empty cells as failures, which conflates absent with unusable. Distinct minus converts subtracts two counts that answer different questions. Present alone is not a gap at all.

## Step 2. Read profile B

B looks better on every count that rose. Find the one count that fell.

In [6]:
kit.flow(["read profile A", "read profile B", "compare them", "say which and why"], lit=1)

In [7]:
# TODO 2. Which count fell between A and B?
#   a) present
#   b) distinct
#   c) converts
#   d) none of them fell
fell = "distinct"
print("B amount:", B["amount"])
print("the count that fell:", fell)

B amount: {'present': 50, 'converts': 50, 'distinct': 42}
the count that fell: distinct


In [8]:
kit.check("present rose to fifty in B", B["amount"]["present"] == 50)
kit.check("converts rose to fifty in B", B["amount"]["converts"] == 50)
kit.check("distinct fell", B["amount"]["distinct"] < A["amount"]["distinct"],
          f'{A["amount"]["distinct"]} down to {B["amount"]["distinct"]}')
kit.check("and you named it", fell == "distinct")

Present and converts both rose, which is what makes B look like progress. Saying nothing fell is the reading that ships a coerced file, and it is the exact failure this exercise exists to catch.

## Step 3. Compare them

Both profiles describe the same fifty orders. Only one of them still knows what the source sent.

In [9]:
kit.flow(["read profile A", "read profile B", "compare them", "say which and why"], lit=2)

In [10]:
# TODO 3. Which set difference finds what B threw away?
#   a) {r["amount"] for r in coerced} - {r["amount"] for r in orders}
#   b) {r["amount"] for r in orders} & {r["amount"] for r in coerced}
#   c) {r["amount"] for r in orders} - {r["amount"] for r in coerced}
#   d) {r["amount"] for r in orders} | {r["amount"] for r in coerced}
values_lost = {r["amount"] for r in orders} - {r["amount"] for r in coerced}
print(len(values_lost), "raw amount values exist in A and not in B")
print(sorted(values_lost))

5 raw amount values exist in A and not in B
['', '12,400', '24 500', 'Rs 8000', 'twelve']


In [11]:
kit.check("five raw values are gone from B", len(values_lost) == 5,
          "the two empty cells shared one raw value")
kit.check("the word twelve is one of them", "twelve" in values_lost)

Reversing the difference finds what B invented rather than what it lost. An intersection finds what survived. A union finds everything either file ever held.

## Step 4. Say which, and why

Write the sentence. It names the file you would compute on and the single count that decided it.

In [12]:
kit.flow(["read profile A", "read profile B", "compare them", "say which and why"], lit=3)

In [13]:
# TODO 4. What ends that sentence?
#   a) it has fewer rows to clean
#   b) it looks less complete
#   c) B has more converts
#   d) distinct fell in B, and that is the only warning
verdict = "A, because " + "distinct fell in B, which is the only sign that six real values were replaced"
print(verdict)

A, because distinct fell in B, which is the only sign that six real values were replaced


In [14]:
kit.check("your verdict picks A", verdict.startswith("A,"))
kit.check("and it names the count that decided it", "distinct" in verdict)

## What to post

Post one line with the four letters, then the two numbers:

```
1a 2b 3c 4d
A distinct 46, B distinct 41
```

Then one sentence on what you would say to the colleague who produced B and thought they had helped.

In [15]:
kit.flow(["read profile A", "read profile B", "compare them", "say which and why"], lit=3, title="the notebook, end to end")
kit.check_summary()